# ST 554 Homework 9
---
Authored by: Jamie Loring

Collaborator: Dr. Justin Post (code from lecture videos & slides)

In [18]:
#importing required modules
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor, RandomForestRegressor
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

## Reading in Data
For this assignment, I chose to use a dataset on Florida real estate properties sold in 2026. I downloaded this dataset from Kaggle, and it is linked [here](https://www.kaggle.com/datasets/kanchana1990/florida-real-estate-sold-dataset-2026). The data file will also be available in my Github, titled `florida_real_estate_sold_properties_utlimate.csv`.

### Creating a Spark Session
The code below creates a spark session for use with reading in our data and building our models.

In [2]:
spark = SparkSession.builder.master('local[*]').appName('my_app').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 20:24:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


The code below only allows error messages to print out going forward.

In [3]:
spark.sparkContext.setLogLevel("ERROR")

### Importing Data
The code below reads in our data using `pandas`. The `.dropna()` method is also used to drop rows where `NaN` values are present within certain variables. I made sure that the variables supplied do not result in exopnential data loss. Finally, the `.fillna()` method is used with context clues to fill missing values within certain variables. For the `stories` variable, `NaN` will be changed to 1.0. For the `garage` variable, `NaN` will be changed to 0.

In [4]:
house_data = pd.read_csv("florida_real_estate_sold_properties_ultimate.csv") \
             .dropna(subset=['sqft', 'year_built', 'beds', 'baths_full'])

house_data['stories'] = house_data['stories'].fillna(1.0)
house_data['garage'] = house_data['garage'].fillna(0.0)

house_data.head()

,type,sub_type,listPrice,lastSoldPrice,sqft,stories,beds,baths,baths_full,baths_full_calc,garage,year_built,zip,sanitized_text
0,single_family,NaN,630000.0,605000,2274.0,1.0,2.0,3.0,2.0,2.0,2.0,2007.0,33446.0,"Beautiful 2 Bedroom + Den, 2.5 Bath Home - Mov..."
1,single_family,NaN,289000.0,285000,2170.0,1.0,3.0,2.0,2.0,2.0,2.0,1980.0,33876.0,Welcome to Florida living at its best! This 3-...
2,condos,condo,449000.0,425000,1722.0,1.0,3.0,2.0,2.0,2.0,2.0,2016.0,33913.0,Best Value in Casella and priced to sell... St...
3,single_family,NaN,599000.0,596000,1699.0,1.0,3.0,3.0,3.0,3.0,0.0,1952.0,33009.0,"Beautifully renovated 3-bedroom, 3-bathroom ho..."
4,single_family,NaN,173500.0,165000,640.0,1.0,1.0,1.0,1.0,1.0,0.0,1971.0,32118.0,Experience the ultimate beachfront lifestyle i...


Now, we convert our `pandas dataframe` to a `spark` SQL style dataframe.

In [5]:
FL_houses = spark.createDataFrame(house_data)
FL_houses.show(5)

+-------------+--------+---------+-------------+------+-------+----+-----+----------+---------------+------+----------+-------+--------------------+
|         type|sub_type|listPrice|lastSoldPrice|  sqft|stories|beds|baths|baths_full|baths_full_calc|garage|year_built|    zip|      sanitized_text|
+-------------+--------+---------+-------------+------+-------+----+-----+----------+---------------+------+----------+-------+--------------------+
|single_family|     NaN| 630000.0|       605000|2274.0|    1.0| 2.0|  3.0|       2.0|            2.0|   2.0|    2007.0|33446.0|Beautiful 2 Bedro...|
|single_family|     NaN| 289000.0|       285000|2170.0|    1.0| 3.0|  2.0|       2.0|            2.0|   2.0|    1980.0|33876.0|Welcome to Florid...|
|       condos|   condo| 449000.0|       425000|1722.0|    1.0| 3.0|  2.0|       2.0|            2.0|   2.0|    2016.0|33913.0|Best Value in Cas...|
|single_family|     NaN| 599000.0|       596000|1699.0|    1.0| 3.0|  3.0|       3.0|            3.0|   0.

## Splitting the Data, Metrics, and Models
This section involves:
- splitting the data into a training and test set using spark MLlib
- choosing and describing a metric for judging the fitted models
- fitting three different classes of models and describing them

### Splitting the Data
The code below uses spark MLlib to split the data into a training and test set. This is done by using the `.randomSplit()` method on a `spark` SQL style dataframe.

In [6]:
train, test = FL_houses.randomSplit([0.8,0.2], seed = 44)
print('Number of training observations:', train.count())
print('Number of test observations:', test.count())

Number of training observations: 8017
Number of test observations: 2064


### Choosing a Metric
For this assignment, I will choose the root mean square error (RMSE) for judging the models. RMSE is a very common metric and is easy to interpret. The lower the RMSE, the better the model fit. The mean square error (MSE) is the average of the difference between observed values and values predicted by the model. We want to minimze MSE because we want our model predictions to be close to what is observed! As such, the RMSE is simply the square-root of the MSE.

### Classes of Models
Since I am interested in using `lastSoldPrice` as the response, I will fit three different classes of *regression* models. The following models will be fit:
- A **linear regression model**
- A **decision tree regression model**
- A **random forest regression model**

## Model Fitting
In this section, we use Spark MLlib to fit the three different classes of models described previously to the training data. A pipeline in `pyspark` will be set up for each model. Cross validation will be used to choose the best model for each model type. These best models will be compared using RMSE; but remember, the test data will be used to select the best model overall!

### Model 1: Linear Regression Model
The first model we will fit is a linear regression model that uses the following features to predict log(`lastSoldPrice`):
- `sqft`
- a multi-story indicator variable that will be calculated using the `stories` variable
    - will be called `multi_story_ind`
- `year_built`

**Note:** This is the model that will feature 4 transformations in the pipeline!

We first need to create our multi-story indicator variable. This can be done using `Binarizer()`. We will use a threshold of 1.5 since anything lower would clearly indicate that the home is not multi-story, whereas anything larger would clearly indicate that the home is multi-story.

In [7]:
binaryTrans1 = Binarizer(threshold = 1.5, inputCol = "stories", outputCol = "multi_story_ind")
binaryTrans1.transform(FL_houses).show(5)

+-------------+--------+---------+-------------+------+-------+----+-----+----------+---------------+------+----------+-------+--------------------+---------------+
|         type|sub_type|listPrice|lastSoldPrice|  sqft|stories|beds|baths|baths_full|baths_full_calc|garage|year_built|    zip|      sanitized_text|multi_story_ind|
+-------------+--------+---------+-------------+------+-------+----+-----+----------+---------------+------+----------+-------+--------------------+---------------+
|single_family|     NaN| 630000.0|       605000|2274.0|    1.0| 2.0|  3.0|       2.0|            2.0|   2.0|    2007.0|33446.0|Beautiful 2 Bedro...|            0.0|
|single_family|     NaN| 289000.0|       285000|2170.0|    1.0| 3.0|  2.0|       2.0|            2.0|   2.0|    1980.0|33876.0|Welcome to Florid...|            0.0|
|       condos|   condo| 449000.0|       425000|1722.0|    1.0| 3.0|  2.0|       2.0|            2.0|   2.0|    2016.0|33913.0|Best Value in Cas...|            0.0|
|single_fa

Next, we use the `SQLTransformer()` to select the columns that we want and rename the response, the log of `lastSoldPrice`, as `label`.

In [8]:
sqlTrans1 = SQLTransformer(
    statement = """
                SELECT sqft, multi_story_ind, year_built, log(lastSoldPrice) as label FROM __THIS__
                """
            )

sqlTrans1.transform(
    binaryTrans1.transform(FL_houses)
).show(5)

+------+---------------+----------+------------------+
|  sqft|multi_story_ind|year_built|             label|
+------+---------------+----------+------------------+
|2274.0|            0.0|    2007.0|13.312983737012978|
|2170.0|            0.0|    1980.0|12.560244459250788|
|1722.0|            0.0|    2016.0|12.959844447906553|
|1699.0|            0.0|    1952.0|13.297995946047486|
| 640.0|            0.0|    1971.0|12.013700752882718|
+------+---------------+----------+------------------+
only showing top 5 rows


Continuing on, we need to put our model features into a single column called `features`. We can do this with `VectorAssembler()`.

In [9]:
assembler1 = VectorAssembler(
                inputCols = ["sqft", "multi_story_ind", "year_built"],
                outputCol = "features",
                handleInvalid = "keep"
             )

assembler1.transform(
    sqlTrans1.transform(
        binaryTrans1.transform(FL_houses)
    )
).show(5)

+------+---------------+----------+------------------+-------------------+
|  sqft|multi_story_ind|year_built|             label|           features|
+------+---------------+----------+------------------+-------------------+
|2274.0|            0.0|    2007.0|13.312983737012978|[2274.0,0.0,2007.0]|
|2170.0|            0.0|    1980.0|12.560244459250788|[2170.0,0.0,1980.0]|
|1722.0|            0.0|    2016.0|12.959844447906553|[1722.0,0.0,2016.0]|
|1699.0|            0.0|    1952.0|13.297995946047486|[1699.0,0.0,1952.0]|
| 640.0|            0.0|    1971.0|12.013700752882718| [640.0,0.0,1971.0]|
+------+---------------+----------+------------------+-------------------+
only showing top 5 rows


We are now ready to set up a pipeline for this model and fit a multiple linear regression (MLR) model. A few important notes:
- The `Pipeline()` function from `pyspark.ml` will be used to set up our sequence of transformations and estimators
- Since `LinearRegression()` does regularized regression, we will provide a list of values for both `regParam` and `elasticNetParam` so that cross-validation can be used to select the best model!

In [10]:
# set up instance of our linear regression
lr = LinearRegression()

# build parameter grid
paramGrid1 = ParamGridBuilder() \
             .addGrid(lr.regParam, [0, 0.25, 0.5, 0.75, 1]) \
             .addGrid(lr.elasticNetParam, [0, 0.5, 0.75, 0.9, 1]) \
             .build()

# create pipeline
pipeline1 = Pipeline(stages = [binaryTrans1, sqlTrans1, assembler1, lr])

# set up cross validation with pipeline
crossval1 = CrossValidator(estimator = pipeline1,
                           estimatorParamMaps = paramGrid1,
                           evaluator = RegressionEvaluator(metricName = 'rmse'),
                           numFolds = 5,
                           seed = 44)

# fit the model using cross-validation to choose the best model
cvModel1 = crossval1.fit(train)

Although the `.bestModel` attribute will store the best model as a result of cross-validation by default, we want to look to see what model is the best. The code below prints the RMSE for the model fit with its corresponding parameters, `regParam` and `elasticNetParam`. I then sort this output in ascending order so we can see the lowest RMSE first (along with its corresponding parameters.

In [11]:
my_list1 = []

for i in range(len(paramGrid1)):
    my_list1.append([cvModel1.avgMetrics[i], paramGrid1[i].values()])
    
my_list1.sort(key=lambda x: x[0]) #specifies sorting by avgMetrics rather than the dict_values
my_list1

[[0.6343837456303907, dict_values([0.0, 1.0])],
 [0.6343837456303923, dict_values([0.0, 0.75])],
 [0.6343837456303939, dict_values([0.0, 0.5])],
 [0.6343837456303946, dict_values([0.0, 0.9])],
 [0.634383745630395, dict_values([0.0, 0.0])],
 [0.6458423606674886, dict_values([0.25, 0.0])],
 [0.6575285482882448, dict_values([0.25, 0.5])],
 [0.6635905078052029, dict_values([0.5, 0.0])],
 [0.6672701959580031, dict_values([0.25, 0.75])],
 [0.6746628725355596, dict_values([0.25, 0.9])],
 [0.6796756349328055, dict_values([0.75, 0.0])],
 [0.6803765749580132, dict_values([0.25, 1.0])],
 [0.6932998925009984, dict_values([1.0, 0.0])],
 [0.7061748994118173, dict_values([0.5, 0.5])],
 [0.7454268617753418, dict_values([0.5, 0.75])],
 [0.7601042880521902, dict_values([0.75, 0.5])],
 [0.7785223840507147, dict_values([0.5, 0.9])],
 [0.8060274806039194, dict_values([0.5, 1.0])],
 [0.8118607942586846, dict_values([1.0, 0.5])],
 [0.8216052738428982, dict_values([0.75, 0.75])],
 [0.8216052738428982, dict_va

Thus, the best multiple linear regression model is one with a regularization parameter of 0 and an elastic net parameter of 0.9. We can now print the intercept and coefficients of this best model.

In [12]:
# need to use indexing since the model is the last stage of the pipeline
print('Intercept:', cvModel1.bestModel.stages[-1].intercept)
print('log_sqft, multi_story_ind, year_built coeffs:', cvModel1.bestModel.stages[-1].coefficients)
print('RMSE:', round(my_list1[0][0], 5))

Intercept: 12.560329349272555
log_sqft, multi_story_ind, year_built coeffs: [0.0005833275964945925,-0.022073887276471513,-0.0003728335312881692]
RMSE: 0.63438


### Model 2: Decision Tree Regression Model
The second model we will fit is a decision tree regression model that uses the following features to predict log(`lastSoldPrice`):
- `stories`
- `beds`
- `baths_full`
- `garage`

We start by using the `SQLTransformer()` to select the columns that we want and rename the response, the log of `lastSoldPrice`, as `label`.

In [13]:
sqlTrans2 = SQLTransformer(
    statement = """
                SELECT stories, beds, baths_full, garage, log(lastSoldPrice) as label FROM __THIS__
                """
            )

sqlTrans2.transform(FL_houses).show(5)

+-------+----+----------+------+------------------+
|stories|beds|baths_full|garage|             label|
+-------+----+----------+------+------------------+
|    1.0| 2.0|       2.0|   2.0|13.312983737012978|
|    1.0| 3.0|       2.0|   2.0|12.560244459250788|
|    1.0| 3.0|       2.0|   2.0|12.959844447906553|
|    1.0| 3.0|       3.0|   0.0|13.297995946047486|
|    1.0| 1.0|       1.0|   0.0|12.013700752882718|
+-------+----+----------+------+------------------+
only showing top 5 rows


Next, we need to put our model features into a single column called `features`. We can do this with `VectorAssembler()`.

In [14]:
assembler2 = VectorAssembler(
                inputCols = ["stories", "beds", "baths_full", "garage"],
                outputCol = "features",
                handleInvalid = "keep"
             )

assembler2.transform(
    sqlTrans2.transform(FL_houses)
).show(5)

+-------+----+----------+------+------------------+-----------------+
|stories|beds|baths_full|garage|             label|         features|
+-------+----+----------+------+------------------+-----------------+
|    1.0| 2.0|       2.0|   2.0|13.312983737012978|[1.0,2.0,2.0,2.0]|
|    1.0| 3.0|       2.0|   2.0|12.560244459250788|[1.0,3.0,2.0,2.0]|
|    1.0| 3.0|       2.0|   2.0|12.959844447906553|[1.0,3.0,2.0,2.0]|
|    1.0| 3.0|       3.0|   0.0|13.297995946047486|[1.0,3.0,3.0,0.0]|
|    1.0| 1.0|       1.0|   0.0|12.013700752882718|[1.0,1.0,1.0,0.0]|
+-------+----+----------+------+------------------+-----------------+
only showing top 5 rows


We are now ready to set up a pipeline for this model and fit a random forest regression model. A few important notes:
- The `Pipeline()` function from `pyspark.ml` will be used to set up our sequence of transformations and estimators
- A list of values will be provided for the `maxDepth` and `minInstancesPerNode` options so that cross-validation can be used to select the best model!

In [15]:
# set up instance of our decision tree
dt = DecisionTreeRegressor(seed = 44)

# build parameter grid
paramGrid2 = ParamGridBuilder() \
             .addGrid(dt.maxDepth, [2, 3, 4, 5, 6, 7, 8]) \
             .addGrid(dt.minInstancesPerNode, [2, 3, 5, 10, 20]) \
             .build()

# create pipeline
pipeline2 = Pipeline(stages = [sqlTrans2, assembler2, dt])

# set up cross validation with pipeline
crossval2 = CrossValidator(estimator = pipeline2,
                           estimatorParamMaps = paramGrid2,
                           evaluator = RegressionEvaluator(metricName = 'rmse'),
                           numFolds = 5,
                           seed = 44)

# fit the model using cross-validation to choose the best model
cvModel2 = crossval2.fit(train)

Although the `.bestModel` attribute will store the best model as a result of cross-validation by default, we want to look to see what model is the best. The code below prints the RMSE for the model fit with its corresponding parameters, `maxDepth` and `minInstancesPerNode`. I then sort this output in ascending order so we can see the lowest RMSE first (along with its corresponding parameters.my_list1 = []

for i in range(len(paramGrid1)):
    my_list1.append([cvModel1.avgMetrics[i], paramGrid1[i].values()])
    
my_list1.sort(key=lambda x: x[0]) #specifies sorting by avgMetrics rather than the dict_values
my_list1

In [16]:
my_list2 = []

for i in range(len(paramGrid2)):
    my_list2.append([cvModel2.avgMetrics[i], paramGrid2[i].values()])
    
my_list2.sort(key=lambda x: x[0]) #specifies sorting by avgMetrics rather than the dict_values
my_list2

[[0.6274233093600641, dict_values([7, 5])],
 [0.6276732581145156, dict_values([6, 5])],
 [0.6278775015576186, dict_values([8, 5])],
 [0.6280277189395966, dict_values([5, 5])],
 [0.6285184146360159, dict_values([7, 10])],
 [0.6288767806497781, dict_values([6, 3])],
 [0.6289251227380371, dict_values([8, 10])],
 [0.6290354642861244, dict_values([5, 10])],
 [0.6292012865687174, dict_values([7, 3])],
 [0.6292738161943516, dict_values([6, 10])],
 [0.6293434461451165, dict_values([5, 3])],
 [0.6298917165627447, dict_values([5, 2])],
 [0.6299851734513208, dict_values([7, 2])],
 [0.6301310059090974, dict_values([5, 20])],
 [0.6301567112768602, dict_values([6, 20])],
 [0.6302119162922928, dict_values([8, 3])],
 [0.6302755508853676, dict_values([8, 20])],
 [0.6303127344458137, dict_values([7, 20])],
 [0.6310882674045214, dict_values([6, 2])],
 [0.6322662419541594, dict_values([8, 2])],
 [0.6330375376940448, dict_values([4, 20])],
 [0.6336686728682911, dict_values([4, 5])],
 [0.6338731315173469, d

Thus, the best decision tree regression model is one with a max depth of 7 and and 5 minimum instances per node. We will now print the total number of nodes in the decision tree.

In [17]:
# need to use indexing since the model is the last stage of the pipeline
print('Total Number of Nodes:', cvModel2.bestModel.stages[-1].numNodes)
print('RMSE:', round(my_list2[0][0], 5))

Total Number of Nodes: 145
RMSE: 0.62742


### Model 3: Random Forest Regression Model
The third and final model we will fit is a random forest regression model that uses the following features to predict log(`lastSoldPrice`):
- an indicator variable that takes on a value of 1 for single family homes and 0 otherwise
    - calculated using the `type` variable
    - will be called `sgl_fmily_ind`
- the log of the `listPrice`
- `sqft`
- `beds`
- `baths`
    - this is the total number of all baths, including half-baths